# Задание 2.2. Настройка среды Julia и решение научной задачи

**Выполнил(а):** Ерохина Анастасия
**Группа:** ИВТ 2.1

**Тема:** Развёртывание среды Julia в VS Code и решение задачи классификации цветов Ирисов методом K-ближайших соседей (KNN).

## Настройка среды

1. Установлена Julia версии 1.13.0.
2. Установлен Visual Studio Code с расширением Julia.
3. Создан файл MY.ipynb, выбрано ядро Julia 1.13.0.
4. Через PythonCall.jl + CondaPkg установлены scikit-learn и PythonCall.

## Шаг 0. Установка пакетов

Устанавливаются библиотеки PythonCall.jl и CondaPkg.jl в локальное окружение проекта.

In [6]:
using CondaPkg
CondaPkg.add("scikit-learn")

    CondaPkg Found dependencies: C:\Users\pigeo\OneDrive\Desktop\CondaPkg.toml
    CondaPkg Found dependencies: C:\Users\pigeo\.julia\packages\CondaPkg\lKlVY\CondaPkg.toml
    CondaPkg Found dependencies: C:\Users\pigeo\.julia\packages\PythonCall\x9Db0\CondaPkg.toml
    CondaPkg Initialising pixi
             │ C:\Users\pigeo\.julia\artifacts\6cd2217b3e33a22947fb8d4194f79fd44c7dc25f\bin\pixi.exe
             │ init
             │ --format pixi
             └ C:\Users\pigeo\OneDrive\Desktop\.CondaPkg
Created C:\Users\pigeo\OneDrive\Desktop\.CondaPkg\pixi.toml
    CondaPkg Wrote C:\Users\pigeo\OneDrive\Desktop\.CondaPkg\pixi.toml
             │ [dependencies]
             │ openssl = ">=3, <3.6"
             │ scikit-learn = "*"
             │ 
             │     [dependencies.python]
             │     version = ">=3.10,!=3.14.0,!=3.14.1,<4"
             │     build = "*cp*"
             │     channel = "conda-forge"
             │ 
             │ [workspace]
             │ name = ".Con

In [7]:
import Pkg
Pkg.activate(".")
Pkg.add(["PythonCall", "CondaPkg"])

  Activating project at `c:\Users\pigeo\OneDrive\Desktop`
     Project No packages added to or removed from `C:\Users\pigeo\OneDrive\Desktop\Project.toml`
    Manifest No packages added to or removed from `C:\Users\pigeo\OneDrive\Desktop\Manifest.toml`


## Шаг 1. Подключение библиотек

Используется библиотека **PythonCall.jl** для вызова Python-библиотеки `scikit-learn` из Julia. Датасет Iris встроен в scikit-learn и загружается функцией `load_iris()`.

In [8]:
using PythonCall
sklearn = pyimport("sklearn")

datasets = sklearn.datasets
iris = datasets.load_iris()
X = iris["data"]
y = iris["target"]

println("Размер: ", size(pyconvert(Array, X)))
println("Классы: ", unique(pyconvert(Array, y)))

Размер: (150, 4)
Классы: [0, 1, 2]


## Шаг 2. Разделение данных на обучающую и тестовую выборки

Данные делятся в пропорции 70/30: 105 объектов - для обучения, 45 - для проверки. Параметр `random_state=42` обеспечивает воспроизводимость.

In [9]:
model_selection = sklearn.model_selection

X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y; test_size=0.3, random_state=42
)

println("Обучающая выборка: ", size(pyconvert(Array, X_train)))
println("Тестовая выборка:  ", size(pyconvert(Array, X_test)))

Обучающая выборка: (105, 4)
Тестовая выборка:  (45, 4)


## Шаг 3. Построение модели KNN (k=5)

Метод K-ближайших соседей: для нового объекта находится 5 ближайших из обучающей выборки, и по большинству голосов определяется класс.

In [10]:
model = sklearn.neighbors.KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
println("Предсказания: ", pyconvert(Array, y_pred))

Предсказания: [1, 0, 2, 1, 1, 0, 1, 2, 1, 1, 2, 0, 0, 0, 0, 1, 2, 1, 1, 2, 0, 2, 0, 2, 2, 2, 2, 2, 0, 0, 0, 0, 1, 0, 0, 2, 1, 0, 0, 0, 2, 1, 1, 0, 0]


## Шаг 4. Оценка точности

Сравниваются предсказанные и истинные классы тестовой выборки. Точность - это доля правильных ответов.

In [11]:
acc = sklearn.metrics.accuracy_score(y_test, y_pred)
acc_val = pyconvert(Float64, acc)

println("Точность: ", round(acc_val * 100, digits=2), " %")

cm = sklearn.metrics.confusion_matrix(y_test, y_pred)
println("Матрица ошибок:\n", pyconvert(Array, cm))

Точность: 100.0 %
Матрица ошибок:
[19 0 0; 0 13 0; 0 0 13]


## Вывод

В ходе работы освоено развёртывание среды Julia в VS Code. Задача классификации цветов Ирисов методом KNN (k=5) решена с точностью 100 % на тестовой выборке.